<a href="https://colab.research.google.com/github/oliviacomisso-a11y/DRP-Fall-2026/blob/main/DRP_TASK_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def c(n):
    if n <= 1:
        return 0
    if n == 2:
        return 1
    # Euler-Mascheroni constant is approximately 0.5772156649
    return 2.0 * (np.log(n - 1) + 0.5772156649) - (2.0 * (n - 1) / n)

class ExNode:
    def __init__(self, size):
        self.size = size
        self.node_type = 'external'

class InNode:
    def __init__(self, left, right, split_att, split_value):
        self.left = left
        self.right = right
        self.split_att = split_att
        self.split_value = split_value
        self.node_type = 'internal'

def iTree(X, e, l):
    n_samples, n_features = X.shape
    if e >= l or n_samples <= 1:
        return ExNode(size=n_samples)
    else:
        Q = list(range(n_features))

        valid_attributes = [q for q in Q if np.min(X[:, q]) < np.max(X[:, q])]
        if not valid_attributes:
            return ExNode(size=n_samples)

        q = np.random.choice(valid_attributes)

        min_val = np.min(X[:, q])
        max_val = np.max(X[:, q])
        p = np.random.uniform(min_val, max_val)

        left_mask = X[:, q] < p
        X_l = X[left_mask]

        X_r = X[~left_mask]

        return InNode(
            left=iTree(X_l, e + 1, l),
            right=iTree(X_r, e + 1, l),
            split_att=q,
            split_value=p
        )

def iForest(X, t, psi):

    Forest = []
    l = int(np.ceil(np.log2(psi)))
    n_samples = X.shape[0]

    for i in range(t):
        indices = np.random.choice(n_samples, size=min(psi, n_samples), replace=False)
        X_prime = X[indices]

        tree = iTree(X_prime, 0, l)
        Forest.append(tree)

    return Forest

def PathLength(x, T, e=0):

    if T.node_type == 'external':
        return e + c(T.size)
    a = T.split_att

    if x[a] < T.split_value:
        return PathLength(x, T.left, e + 1)
    else:
        return PathLength(x, T.right, e + 1)

In [ ]:
def compute_anomaly_scores(X, forest, psi):
    """
    Computes anomaly scores for every row in X based on the forest.
    Score s(x, psi) = 2^(- E(h(x)) / c(psi))
    """
    n_samples = X.shape[0]
    avg_path_lengths = np.zeros(n_samples)

    for i in range(n_samples):
        path_lengths = [PathLength(X[i], tree, 0) for tree in forest]
        avg_path_lengths[i] = np.mean(path_lengths)

    c_psi = c(psi)
    if c_psi == 0:
        return np.zeros(n_samples)

    scores = 2.0 ** (-avg_path_lengths / c_psi)
    return scores, avg_path_lengths

features = ['V257', 'V246', 'V244', 'V242', 'V201']
test_data_cleaned = df[features + ['isFraud']].dropna().head(1000)
X_test = test_data_cleaned[features].values
y_test = test_data_cleaned['isFraud'].values

t = 50
psi = 129096
forest = iForest(X_test, t, psi)

scores, paths = compute_anomaly_scores(X_test, forest, psi)
test_data_cleaned['AnomalyScore'] = scores

print("Average anomaly score for Non-Fraud rows:", test_data_cleaned[test_data_cleaned['isFraud'] == 0]['AnomalyScore'].mean())
print("Average anomaly score for Fraudulent rows:", test_data_cleaned[test_data_cleaned['isFraud'] == 1]['AnomalyScore'].mean())

display(test_data_cleaned[['isFraud', 'AnomalyScore']].head(10))